# Task 1-e — 길이 편향 통제 후 HaluEval 재평가

## 배경

직전 실험에서 HaluEval 세 도메인 모두 응답 길이만으로도 높은 AUROC(QA 0.975, Summary 0.756,
Dialogue 0.734)를 얻을 수 있음이 확인되었고, 판정기(MiniCheck, DeBERTa)의 AUROC는 세 도메인
모두에서 이 길이 기준선을 밑돌거나 근소하게만 앞섰다. 즉 직전 실험의 판정기 성능 하락이
판정기의 일반화 실패인지, HaluEval 데이터셋 자체의 길이 편향 때문인지 구분되지 않았다.

## 이 실험의 방법

정상 응답과 환각 응답의 길이 분포가 겹치는 예시만 선별하여 재평가함으로써 길이 신호를
제거한다. 구체적으로 각 쌍(정상, 환각)에서 두 응답의 길이 비율이 일정 범위 안에 드는
쌍만 남긴다. 이렇게 걸러진 표본에서는 길이만으로 구분할 수 없으므로, 판정기의 AUROC가
여전히 높게 유지된다면 그것은 실제 의미 판단에서 비롯된 것으로 볼 수 있다.

## 사전 정의 기준

- 길이 통제 성공 기준: 필터링 후 표본에서 길이 기준선 AUROC가 0.60 미만으로 낮아질 것
- 판정기 유효성 기준: 길이 통제 후에도 AUROC 0.70 이상을 유지하면 해당 도메인에서
  판정기의 성능이 길이 편향이 아닌 실제 판단 능력에서 온다고 본다.

## 1. 환경 준비 (직전 실험과 이어서 실행 가능)

In [ ]:
!pip -q install sentencepiece scikit-learn pandas matplotlib nltk
!git clone --depth 1 https://github.com/RUCAIBox/HaluEval.git 2>/dev/null || echo already cloned
!pip -q install "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main" 2>&1 | tail -2

import nltk
for pkg in ['punkt', 'punkt_tab']:
    try: nltk.download(pkg, quiet=True)
    except Exception as e: print(pkg, 'download 실패:', e)

import json, time, random, gc
import numpy as np, pandas as pd, torch, transformers
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| device', DEVICE)

## 2. 데이터 적재 — 쌍 구조를 유지한 채 로드

In [ ]:
BASE = 'HaluEval/data'

def load_raw_pairs(path, ref_key, right_key, hallu_key, n):
    rows = []
    with open(path) as f:
        lines = f.readlines()
    random.Random(SEED).shuffle(lines)
    for line in lines[:n]:
        d = json.loads(line)
        ref = d[ref_key] if isinstance(ref_key, str) else ref_key(d)
        rows.append({'reference': ref, 'right': d[right_key], 'hallu': d[hallu_key]})
    return pd.DataFrame(rows)

# 필터링으로 표본이 줄어들 것을 감안하여 원본을 더 크게 뽑는다
N_PER_DOMAIN = 500

qa = load_raw_pairs(f'{BASE}/qa_data.json', 'knowledge', 'right_answer', 'hallucinated_answer', N_PER_DOMAIN)
qa['domain'] = 'QA'
summ = load_raw_pairs(f'{BASE}/summarization_data.json', 'document', 'right_summary', 'hallucinated_summary', N_PER_DOMAIN)
summ['domain'] = 'Summary'
dialog = load_raw_pairs(
    f'{BASE}/dialogue_data.json',
    lambda d: d['knowledge'] + '\n' + d['dialogue_history'],
    'right_response', 'hallucinated_response', N_PER_DOMAIN)
dialog['domain'] = 'Dialogue'

pairs = pd.concat([qa, summ, dialog], ignore_index=True)
pairs['len_right'] = pairs.right.str.len()
pairs['len_hallu'] = pairs.hallu.str.len()
pairs['len_ratio'] = pairs.len_hallu / pairs.len_right.clip(lower=1)
print(f'원본 쌍: {len(pairs):,}개')
pairs.groupby('domain').len_ratio.describe()[['mean','50%','min','max']].round(2)

## 3. 길이 비율 기준 필터링

쌍의 길이 비율(환각/정상)이 [1/R, R] 범위 안에 드는 쌍만 남긴다.
R을 좁힐수록 길이 신호는 약해지지만 표본이 줄어든다. 두 값을 시도하여 트레이드오프를 확인한다.

In [ ]:
def build_examples(filtered_pairs):
    """필터링된 쌍에서 정상/환각 각각을 독립 예시로 전개."""
    rows = []
    for r in filtered_pairs.itertuples():
        rows.append({'reference': r.reference, 'claim': r.right, 'y': 0, 'domain': r.domain})
        rows.append({'reference': r.reference, 'claim': r.hallu, 'y': 1, 'domain': r.domain})
    df = pd.DataFrame(rows)
    df['claim_len'] = df.claim.str.len()
    return df

for R in [3.0, 1.5]:
    mask = (pairs.len_ratio >= 1/R) & (pairs.len_ratio <= R)
    kept = pairs[mask]
    tmp = build_examples(kept)
    print(f'--- R={R} (비율 {1/R:.2f}~{R:.2f}배 허용) ---')
    for d, g in tmp.groupby('domain'):
        if g.y.nunique() < 2: continue
        a = roc_auc_score(g.y, g.claim_len)
        print(f'  {d:10} 유지된 쌍 {mask[pairs.domain==d].sum():4}개   길이기준선 AUROC {a:.3f}')
    print()

In [ ]:
R = 1.5   # 위 결과를 보고 확정. 길이기준선이 0.60 미만으로 떨어지는 값을 우선한다
mask = (pairs.len_ratio >= 1/R) & (pairs.len_ratio <= R)
kept = pairs[mask].copy()
df = build_examples(kept)

print(f'필터링 후 표본: {len(df):,}개 예시 ({kept.groupby("domain").size().to_dict()} 쌍)')
print('\n=== 필터링 후 길이 기준선 AUROC (0.60 미만이 목표) ===')
for d, g in df.groupby('domain'):
    if g.y.nunique() < 2:
        print(f'  {d:10} 표본 부족 — 이 도메인은 결론에서 제외'); continue
    a = roc_auc_score(g.y, g.claim_len)
    verdict = '통제 성공' if a < 0.60 else '길이 신호 잔존 — 해석 주의'
    print(f'  {d:10} n={len(g):4}  길이기준선 AUROC {a:.3f}  → {verdict}')

## 4. MiniCheck 재평가 (길이 통제 표본)

In [ ]:
results, all_scores = [], {}

try:
    from minicheck.minicheck import MiniCheck
    mc = MiniCheck(model_name='flan-t5-large', enable_prefix_caching=False)
    docs, claims = df.reference.tolist(), df.claim.tolist()
    t0 = time.perf_counter()
    _, raw_prob, _, _ = mc.score(docs=docs, claims=claims)
    el = time.perf_counter() - t0
    s = 1.0 - np.asarray(raw_prob)
    all_scores['MiniCheck'] = s
    results.append({'judge': 'MiniCheck-FlanT5-L', 'AUROC': roc_auc_score(df.y, s),
                    'AUPRC': average_precision_score(df.y, s), 'ms_per_item': el*1000/len(df)})
    print(f"MiniCheck  AUROC {results[-1]['AUROC']:.3f}  AUPRC {results[-1]['AUPRC']:.3f}  "
          f"{results[-1]['ms_per_item']:.1f} ms/응답")
    del mc
except Exception as e:
    print(f'[MiniCheck 실패] {type(e).__name__}: {str(e)[:300]}')
finally:
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

## 5. DeBERTa 창분할 재평가 (길이 통제 표본, 동일 방법론)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CKPT = 'microsoft/deberta-large-mnli'
tok = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSequenceClassification.from_pretrained(CKPT).to(DEVICE).eval()
ENT_IDX = [i for i, l in model.config.id2label.items() if l.lower().startswith('entail')][0]

WIN_TOKENS, STRIDE, BATCH = 400, 200, 16

def make_windows(text):
    ids = tok(text, add_special_tokens=False)['input_ids']
    if len(ids) <= WIN_TOKENS: return [text]
    out = []
    for i in range(0, len(ids), STRIDE):
        chunk = ids[i:i+WIN_TOKENS]
        if not chunk: break
        out.append(tok.decode(chunk))
        if i + WIN_TOKENS >= len(ids): break
    return out

@torch.no_grad()
def entail_prob(prem, hyp):
    enc = tok(prem, hyp, truncation=True, max_length=512, padding=True, return_tensors='pt').to(DEVICE)
    return model(**enc).logits.softmax(-1)[:, ENT_IDX].cpu().numpy()

win_cache = {i: make_windows(ref) for i, ref in enumerate(df.reference)}

flat_p, flat_h, owner = [], [], []
for idx, row in enumerate(df.itertuples()):
    for w in win_cache[idx]:
        flat_p.append(w); flat_h.append(row.claim); owner.append(idx)

if DEVICE == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
ent = []
for i in range(0, len(flat_p), BATCH):
    ent.extend(entail_prob(flat_p[i:i+BATCH], flat_h[i:i+BATCH]).tolist())
if DEVICE == 'cuda': torch.cuda.synchronize()
el = time.perf_counter() - t0

ent_max = np.zeros(len(df))
for o, e in zip(owner, ent):
    ent_max[o] = max(ent_max[o], e)
s_deberta = 1.0 - ent_max
all_scores['DeBERTa-window'] = s_deberta

results.append({'judge': 'DeBERTa-MNLI (창분할)', 'AUROC': roc_auc_score(df.y, s_deberta),
                'AUPRC': average_precision_score(df.y, s_deberta), 'ms_per_item': el*1000/len(df)})
print(f"DeBERTa 창분할  AUROC {results[-1]['AUROC']:.3f}  AUPRC {results[-1]['AUPRC']:.3f}  "
      f"{results[-1]['ms_per_item']:.1f} ms/응답")

del model; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

## 6. 종합 판정 — 길이 통제 전후 비교

In [ ]:
BEFORE = {   # 직전 실험(길이 통제 없음) 결과
    'MiniCheck-FlanT5-L':    {'AUROC': 0.704, 'len_baseline_avg': (0.975+0.756+0.734)/3},
    'DeBERTa-MNLI (창분할)': {'AUROC': 0.701, 'len_baseline_avg': (0.975+0.756+0.734)/3},
}
RAGTRUTH_REF = {'MiniCheck-FlanT5-L': 0.814, 'DeBERTa-MNLI (창분할)': 0.744}

res = pd.DataFrame(results)
print('=== 길이 통제 전(도메인 평균) → 후 비교, RAGTruth와 대조 ===\n')
for _, r in res.iterrows():
    b = BEFORE.get(r.judge, {})
    rt = RAGTRUTH_REF.get(r.judge)
    print(f'{r.judge}')
    print(f'  길이 통제 전 (HaluEval, 편향 포함): AUROC {b.get("AUROC", float("nan")):.3f}')
    print(f'  길이 통제 후 (HaluEval, 이번 실험) : AUROC {r.AUROC:.3f}')
    if rt: print(f'  RAGTruth (문장 단위, 참고)         : AUROC {rt:.3f}')
    if r.AUROC > b.get('AUROC', 0) + 0.03:
        print('  → 길이 통제 후 성능이 상승. 직전 실험의 저조한 결과는 길이 편향의 영향으로 설명됨')
    elif r.AUROC < 0.65:
        print('  → 길이 통제 후에도 낮음. 길이 편향과 무관하게 일반화 성능 자체가 제한적')
    else:
        print('  → 길이 통제 후 변화 미미')
    print()

print('=== 도메인별 (표본 확보된 경우만) ===')
for name, s in all_scores.items():
    print(f'\n{name}')
    for d, g in df.groupby('domain'):
        if g.y.nunique() < 2: continue
        idx = g.index
        print(f'  {d:10} AUROC {roc_auc_score(g.y, s[idx]):.3f}  (n={len(g)})')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
names = list(res.judge)
before = [BEFORE.get(n,{}).get('AUROC', np.nan) for n in names]
after  = list(res.AUROC)
x = np.arange(len(names)); w = 0.35
ax.bar(x-w/2, before, w, label='길이 통제 전')
ax.bar(x+w/2, after,  w, label='길이 통제 후')
ax.axhline(0.80, color='r', ls='--', lw=1, label='목표 0.80')
ax.axhline(0.50, color='gray', ls=':', lw=1)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=10, ha='right')
ax.set_ylabel('AUROC'); ax.set_ylim(0,1); ax.legend()
ax.set_title('Effect of length-bias control on HaluEval AUROC')
plt.tight_layout(); plt.show()

In [ ]:
res.to_csv('halueval_length_controlled.csv', index=False)
df.assign(**{n: s for n, s in all_scores.items()}).to_csv('halueval_length_controlled_scores.csv', index=False)
print('저장: halueval_length_controlled.csv, halueval_length_controlled_scores.csv')
res